In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tifffile import TiffFile
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Images du jeu d'entraînement
DATASET_FOLDER = Path("dataset_UNZIPPED/")
train_images_paths = sorted(
    list(DATASET_FOLDER.glob('train/images/*.tif'))
)

features = []

# =====================================================
# EXTRACTION DES CARACTÉRISTIQUES
# =====================================================

for file in train_images_paths:

    # Lecture de l'image TIFF
    with TiffFile(file) as tif:
        image = tif.asarray()

    image = image.astype(float)

    # Les 4 bandes : Blue, Green, Red, NIR
    row = []

    # Statistiques des 4 bandes
    for i in range(4):

        band = image[..., i]
        values = band[np.isfinite(band)]

        row.extend([
            np.mean(values),
            np.std(values),
            np.percentile(values, 5),
            np.percentile(values, 50),
            np.percentile(values, 95)
        ])

    # NDVI : Red = bande 3, NIR = bande 4
    red = image[..., 2]
    nir = image[..., 3]

    denominator = nir + red
    valid = np.isfinite(denominator) & (denominator != 0)

    ndvi = (nir[valid] - red[valid]) / denominator[valid]

    row.extend([
        np.mean(ndvi),
        np.std(ndvi)
    ])

    features.append(row)

features = np.array(features)

# =====================================================
# NORMALISATION
# =====================================================

scaler = StandardScaler()
X = scaler.fit_transform(features)

# =====================================================
# DÉTECTION DES IMAGES ATYPIQUES
# =====================================================

model = IsolationForest(
    contamination=0.001,
    random_state=42
)

predictions = model.fit_predict(X)

# Score : plus il est faible, plus l'image est atypique
scores = model.decision_function(X)

# =====================================================
# RÉSULTATS
# =====================================================

results = pd.DataFrame({
    "Image": [os.path.basename(f) for f in train_images_paths],
    "Outlier": predictions == -1,
    "Score": scores
})

results = results.sort_values("Score")

print(results)

# =====================================================
# AFFICHAGE DES IMAGES OUTLIERS ET NORMALES
# =====================================================

n_images = 5

outlier_indices = np.argsort(scores)[:n_images]
normal_indices = np.argsort(scores)[-n_images:][::-1]

fig, axes = plt.subplots(2, n_images, figsize=(18, 8))

for row, indices, label in [
    (0, outlier_indices, "Images atypiques"),
    (1, normal_indices, "Images normales")
]:

    for col, idx in enumerate(indices):

        # Lecture de l'image TIFF
        with TiffFile(train_images_paths[idx]) as tif:
            image = tif.asarray()

        image = image.astype(float)

        # Affichage RGB : Red, Green, Blue
        rgb = image[..., [2, 1, 0]]

        # Ajustement du contraste
        p2 = np.percentile(rgb, 2)
        p98 = np.percentile(rgb, 98)

        rgb = np.clip(
            (rgb - p2) / (p98 - p2 + 1e-8),
            0,
            1
        )

        axes[row, col].imshow(rgb)

        axes[row, col].set_title(
            f"{train_images_paths[idx].name}\n"
            f"Score : {scores[idx]:.4f}"
        )

        axes[row, col].axis("off")

    axes[row, 0].set_ylabel(label, fontsize=12)

plt.tight_layout()
plt.show()

KeyboardInterrupt: 

In [1]:
# =====================================================
# RÉSULTATS
# =====================================================

print(results)

# =====================================================
# EXPORT DES IMAGES À IGNORER
# =====================================================

outliers = results[results["Outlier"] == True].copy()

outliers["sample_id"] = (
    outliers["Image"]
    .str.extract(r'(\d+)')
    .astype(int)
)

outliers[["sample_id"]].to_csv(
    "local_dataset/ignored_labels.csv",
    index=False
)

print(f"\n{len(outliers)} images à ignorer.")
print("Fichier ignored_labels.csv created.")

NameError: name 'results' is not defined